# Preprocessing & Dataset Preparation

This notebook prepares the dataset for a separate training notebook.

## Responsibilities of this notebook
- Load and clean the raw dataset.
- Perform **deterministic, row-level feature extraction/aggregation** that does not learn from the dataset.
- Split the data into stratified training and test sets.
- Define the reusable preprocessing configuration for numerical and categorical columns.
- Save the prepared train/test datasets and preprocessing metadata.

## 1. Imports and configuration

In [1]:
import json, os, joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder
from pandas.api.types import is_numeric_dtype
from sklearn.model_selection import StratifiedKFold

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = "../Data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
DUMPS_DIR = "../Dumps/"
TARGET = "Churn"
RANDOM_STATE = 42
TEST_SIZE = 0.20

os.makedirs(DUMPS_DIR, exist_ok=True)

## 2. Load and clean the raw data

In [2]:
df = pd.read_csv(DATA_PATH).drop(columns=["customerID"])

# TotalCharges is delivered as text in the raw dataset; convert invalid/blank values to NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Shape of the DataFrame:", df.shape)
display(df.sample(5, random_state=RANDOM_STATE))


Shape of the DataFrame: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
185,Female,0,Yes,No,1,No,No phone service,DSL,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,24.80,24.80,Yes
2715,Male,0,No,No,41,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),25.25,996.45,No
3825,Female,0,Yes,Yes,52,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.35,1031.70,No
1807,Female,0,No,No,1,Yes,No,Fiber optic,No,No,Yes,No,No,No,Month-to-month,No,Electronic check,76.35,76.35,Yes
132,Male,0,No,No,67,Yes,No,DSL,No,No,No,Yes,No,No,Two year,No,Bank transfer (automatic),50.55,3260.10,No


In [3]:
# Remove exact duplicate rows.
print("Duplicate rows before removal:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after duplicate removal:", df.shape)


Duplicate rows before removal: 22
Shape after duplicate removal: (7021, 20)


## 3. Handle the Missing Values in `TotalCharges`

Study `TotalCharges` against `MonthlyCharges` * `tenure` for all customers, and specifically for Customers with `TotalChanges` *= N/A*

In [4]:
def get_missing_pressent():
    missing_percent = df.isnull().mean() * 100
    missing = df.isnull().sum()

    missing_table = pd.DataFrame({
        "Missing Count": missing,
        "Missing %": missing_percent
    })

    print(missing_table.sort_values("Missing %", ascending=False))

get_missing_pressent()

                  Missing Count  Missing %
TotalCharges                 11   0.156673
gender                        0   0.000000
Partner                       0   0.000000
SeniorCitizen                 0   0.000000
Dependents                    0   0.000000
tenure                        0   0.000000
MultipleLines                 0   0.000000
PhoneService                  0   0.000000
OnlineSecurity                0   0.000000
OnlineBackup                  0   0.000000
DeviceProtection              0   0.000000
InternetService               0   0.000000
TechSupport                   0   0.000000
StreamingTV                   0   0.000000
Contract                      0   0.000000
StreamingMovies               0   0.000000
PaperlessBilling              0   0.000000
PaymentMethod                 0   0.000000
MonthlyCharges                0   0.000000
Churn                         0   0.000000


In [5]:
# Inspect the rows where TotalCharges could not be parsed.
missing_total = df[df["TotalCharges"].isna()][
    ["tenure", "MonthlyCharges", "TotalCharges", TARGET]
]

display(missing_total)
print("Missing TotalCharges:", len(missing_total))

,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,NaN,No
753,0,20.25,NaN,No
936,0,80.85,NaN,No
1081,0,25.75,NaN,No
1338,0,56.05,NaN,No
3323,0,19.85,NaN,No
3817,0,25.35,NaN,No
4370,0,20.00,NaN,No
5207,0,19.70,NaN,No
6652,0,73.35,NaN,No


Missing TotalCharges: 11


In [6]:
# Explore the relation between `TotalCharges` and ``, also check if the `ExpectedTotalCharges` is within margen of error
compare_df = df[["MonthlyCharges", "tenure", "TotalCharges"]].copy()
compare_df["ExpectedTotalCharges"] = (compare_df["MonthlyCharges"] * compare_df["tenure"])
compare_df["ChargeDifference"] = (compare_df["TotalCharges"] - compare_df["ExpectedTotalCharges"])

compare_df.sample(10, random_state=RANDOM_STATE)

,MonthlyCharges,tenure,TotalCharges,ExpectedTotalCharges,ChargeDifference
3909,90.20,20,1776.55,1804.00,-27.45
2462,97.95,49,4917.90,4799.55,118.35
2879,116.10,67,7839.85,7778.70,61.15
3761,19.90,57,1115.60,1134.30,-18.70
3745,29.15,1,29.15,29.15,0.00
4482,25.30,18,454.65,455.40,-0.75
2173,19.90,53,1110.05,1054.70,55.35
2476,25.00,61,1501.75,1525.00,-23.25
1566,46.30,15,639.45,694.50,-55.05
1608,90.95,72,6468.60,6548.40,-79.80


In [7]:
# In this dataset, blank TotalCharges occurs for new customers with no accumulated charges.
# Treat those blanks as zero accumulated charges based on that dataset-specific meaning.
df["TotalCharges"] = df["TotalCharges"].fillna(0)
get_missing_pressent()

                  Missing Count  Missing %
gender                        0        0.0
SeniorCitizen                 0        0.0
Partner                       0        0.0
Dependents                    0        0.0
tenure                        0        0.0
PhoneService                  0        0.0
MultipleLines                 0        0.0
InternetService               0        0.0
OnlineSecurity                0        0.0
OnlineBackup                  0        0.0
DeviceProtection              0        0.0
TechSupport                   0        0.0
StreamingTV                   0        0.0
StreamingMovies               0        0.0
Contract                      0        0.0
PaperlessBilling              0        0.0
PaymentMethod                 0        0.0
MonthlyCharges                0        0.0
TotalCharges                  0        0.0
Churn                         0        0.0


## 4. Deterministic feature extraction & aggregation

These features depend only on the values of the current customer. They do **not** use dataset-wide statistics, so they can safely be created before the train/test split.

In [8]:
SERVICE_COLS = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]

# Service usage count: counts active services only.
df["ServiceCount"] = sum(
    (df[col] == "Yes").astype(int)
    for col in SERVICE_COLS
)

In [9]:
# Entertainment/streaming count.
df["StreamingCount"] = (
    (df["StreamingTV"] == "Yes").astype(int)
    + (df["StreamingMovies"] == "Yes").astype(int)
)

In [10]:
# Security/support count.
df["SecuritySupportCount"] = (
    (df["OnlineSecurity"] == "Yes").astype(int)
    + (df["TechSupport"] == "Yes").astype(int)
    + (df["DeviceProtection"] == "Yes").astype(int)
)

In [11]:
# Contract/payment flags.
df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)
df["IsElectronicCheck"] = (df["PaymentMethod"] == "Electronic check").astype(int)

In [12]:
# Household/family indicator.
df["HasFamily"] = (
    (df["Partner"] == "Yes") | (df["Dependents"] == "Yes")
).astype(int)

In [13]:
# Fixed business-rule tenure flag: no dataset statistic is learned.
df["IsNewCustomer"] = (df["tenure"] <= 6).astype(int)

In [14]:
# Derived spending relationship. For tenure=0, use the current monthly charge.
df["AverageMonthlySpend"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

In [15]:
engineered_cols = [
    "ServiceCount",
    "StreamingCount",
    "SecuritySupportCount",
    "IsMonthToMonth",
    "IsElectronicCheck",
    "HasFamily",
    "IsNewCustomer",
    "AverageMonthlySpend",
]

display(df[engineered_cols].head())

,ServiceCount,StreamingCount,SecuritySupportCount,IsMonthToMonth,IsElectronicCheck,HasFamily,IsNewCustomer,AverageMonthlySpend
0,1,0,0,1,1,1,1,29.850000
1,3,0,2,0,0,0,0,55.573529
2,3,0,1,1,0,0,1,54.075000
3,3,0,3,0,0,0,0,40.905556
4,1,0,0,1,1,0,1,75.825000


## 5. Separate features (`X`) and target (`y`)

In [16]:
X = df.drop(columns=TARGET).copy()
y = df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target distribution:")
display(y.value_counts())
display(y.value_counts(normalize=True).rename("proportion"))


X shape: (7021, 27)
y shape: (7021,)
Target distribution:


Churn
No     5164
Yes    1857
Name: count, dtype: int64

Churn
No     0.735508
Yes    0.264492
Name: proportion, dtype: float64

## 6. Train/test Split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")

Train: X=(5616, 27), y=(5616,)
Test:  X=(1405, 27), y=(1405,)


In [18]:
split_summary = pd.DataFrame({
    "Train": y_train.value_counts(normalize=True),
    "Test": y_test.value_counts(normalize=True),
}).rename_axis("Churn")

display(split_summary)

,Train,Test
Churn,,
No,0.735577,0.735231
Yes,0.264423,0.264769


## 7. Features that must be learned from training data

In [19]:
# Spending
median_mon_charges = X_train["MonthlyCharges"].median()

X_train["NewCustomer_HighCharge"] = ((X_train["tenure"] <= 6) &
 (X_train["MonthlyCharges"] > median_mon_charges)).astype(int)

X_test["NewCustomer_HighCharge"] = ((X_test["tenure"] <= 6) &
 (X_test["MonthlyCharges"] > median_mon_charges)).astype(int)

In [20]:
def group_values(feature_name, bins_num, X_train, df_save):
    min_value = X_train[feature_name].min()
    max_value = X_train[feature_name].max()

    bins = np.linspace(
        min_value,
        max_value,
        bins_num + 1
    )

    labels = [
        f"{bins[i]:.0f}-{bins[i+1]:.0f}"
        for i in range(len(bins) - 1)
    ]

    print("Labels for the ranges:", labels)

    name = f"{feature_name}Group"

    df_save[name] = pd.cut(
        df_save[feature_name],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    return name

In [21]:
# Group MonthlyCharge values
group_monthlycharge_train = group_values('MonthlyCharges', 5,X_train, X_train)
group_totalcharge_train = group_values('TotalCharges',5, X_train, X_train)

# Group TotalCharge values
group_monthlycharge_test = group_values('MonthlyCharges', 5, X_train, X_test)
group_totalcharge_train = group_values('TotalCharges',5, X_train, X_test)

Labels for the ranges: ['18-38', '38-58', '58-79', '79-99', '99-119']
Labels for the ranges: ['0-1737', '1737-3474', '3474-5211', '5211-6948', '6948-8685']
Labels for the ranges: ['18-38', '38-58', '58-79', '79-99', '99-119']
Labels for the ranges: ['0-1737', '1737-3474', '3474-5211', '5211-6948', '6948-8685']


In [22]:
# Tenure
median_tenure = X_train["tenure"].median()

X_train["IsOldCustomer"] = (X_train["tenure"] > median_tenure).astype(int)
X_test["IsOldCustomer"] = (X_test["tenure"] > median_tenure).astype(int)

# Group TotalCharge values
group_tenure_train = group_values('tenure',5,X_train, X_train)
group_tenure_test = group_values('tenure',5,X_train, X_test)

Labels for the ranges: ['0-14', '14-29', '29-43', '43-58', '58-72']
Labels for the ranges: ['0-14', '14-29', '29-43', '43-58', '58-72']


In [23]:
engineered_cols_p2 = [
    'NewCustomer_HighCharge',
    'NewCustomer_HighCharge',
    group_monthlycharge_train,
    group_totalcharge_train,
    group_monthlycharge_test,
    group_totalcharge_train,
    'IsOldCustomer',
    group_tenure_train,
    group_tenure_test
]

display(X_train[engineered_cols_p2].head(5))

,NewCustomer_HighCharge,NewCustomer_HighCharge,MonthlyChargesGroup,TotalChargesGroup,MonthlyChargesGroup,TotalChargesGroup,IsOldCustomer,tenureGroup,tenureGroup
2624,1,1,58-79,0-1737,58-79,0-1737,0,0-14,0-14
2645,0,0,79-99,0-1737,79-99,0-1737,0,14-29,14-29
1416,0,0,79-99,5211-6948,79-99,5211-6948,1,58-72,58-72
5760,0,0,58-79,0-1737,58-79,0-1737,0,0-14,0-14
2420,0,0,18-38,0-1737,18-38,0-1737,0,14-29,14-29


## 8. Define the reusable preprocessing configuration

This section **defines** the preprocessing steps but does not fit them.

Processing design:
- Continuous numeric features → median imputation → Yeo-Johnson → standardization.
- Binary numeric features are treated as categorical/binary and are not skew-corrected.
- Categorical features → most-frequent imputation → one-hot encoding.
- `drop="if_binary"` reduces binary categorical variables to one encoded column.
- `handle_unknown="ignore"` keeps inference safe when a new category appears.

In [24]:
def get_binary_categorical_columns(X_df: pd.DataFrame) -> list[str]:
    return [
        col
        for col in X_df.columns
        if not is_numeric_dtype(X_df[col])
        and X_df[col].nunique(dropna=True) == 2
    ]


def get_multi_categorical_columns(X_df: pd.DataFrame) -> list[str]:
    return [
        col
        for col in X_df.columns
        if not is_numeric_dtype(X_df[col])
        and X_df[col].nunique(dropna=True) > 2
    ]


def get_continuous_numeric_columns(X_df: pd.DataFrame) -> list[str]:
    return [
        col
        for col in X_df.columns
        if is_numeric_dtype(X_df[col])
        and X_df[col].nunique(dropna=True) > 2
    ]


def get_binary_numeric_columns(X_df: pd.DataFrame) -> list[str]:
    return [
        col
        for col in X_df.columns
        if is_numeric_dtype(X_df[col])
        and X_df[col].nunique(dropna=True) == 2
    ]


In [25]:
binary_categorical_cols = get_binary_categorical_columns(X_train)
onehot_cols = get_multi_categorical_columns(X_train)
continuous_numeric_cols = get_continuous_numeric_columns(X_train)
binary_numeric_cols = get_binary_numeric_columns(X_train)

# Binary numeric features (e.g. SeniorCitizen) should not be sent through Yeo-Johnson.
numeric_cols = continuous_numeric_cols
categorical_cols = binary_categorical_cols + onehot_cols
binary_numeric_cols


['SeniorCitizen',
 'IsMonthToMonth',
 'IsElectronicCheck',
 'HasFamily',
 'IsNewCustomer',
 'NewCustomer_HighCharge',
 'IsOldCustomer']

In [26]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("yeo_johnson", PowerTransformer(method="yeo-johnson")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        drop="if_binary",
        handle_unknown="ignore",
    )),
])

# Binary numeric columns are passed through unchanged.
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("binary_numeric", "passthrough", binary_numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols),
])

print(preprocessor)


ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('yeo_johnson',
                                                  PowerTransformer()),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'ServiceCount', 'StreamingCount',
                                  'SecuritySupportCount',
                                  'AverageMonthlySpend']),
                                ('binary_numeric', 'passthrough',
                                 ['SeniorCitizen', 'IsMonthToM...
                                                 ('encoder',
                                                  OneHotEncoder(drop='if_binary',
                                                       

## 9. Make the KFold Splits

In [27]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print('KFold Splits: ',skf)

KFold Splits:  StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


## 10. Save prepared data and preprocessing template

The preprocessing object saved here is **unfitted**. This notebook does not train and does not fit preprocessing parameters.

In [28]:
# Save split data separately so the target is not accidentally treated as an input feature.
X_train.to_parquet(os.path.join(DUMPS_DIR, "X_train.parquet"))
X_test.to_parquet(os.path.join(DUMPS_DIR, "X_test.parquet"))
y_train.to_frame(TARGET).to_parquet(os.path.join(DUMPS_DIR, "y_train.parquet"))
y_test.to_frame(TARGET).to_parquet(os.path.join(DUMPS_DIR, "y_test.parquet"))

In [29]:
# Save an UNFITTED preprocessing template for the training notebook.
joblib.dump(preprocessor, os.path.join(DUMPS_DIR, "preprocessor_template.pkl"))

# Save the KFold Splits
joblib.dump(skf, os.path.join(DUMPS_DIR, "skf.pkl"))

['../Dumps/skf.pkl']

In [30]:
feature_metadata = {
    "target": TARGET,
    "continuous_numeric_cols": continuous_numeric_cols,
    "binary_numeric_cols": binary_numeric_cols,
    "binary_categorical_cols": binary_categorical_cols,
    "onehot_cols": onehot_cols,
    "categorical_cols": categorical_cols,
    "engineered_cols": engineered_cols,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
}

with open(os.path.join(DUMPS_DIR, "feature_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(feature_metadata, f, indent=2)

In [31]:
print("Saved:")
for filename in [
    "X_train.parquet",
    "X_test.parquet",
    "y_train.parquet",
    "y_test.parquet",
    "preprocessor_template.pkl",
    "skf.pkl"
    "feature_metadata.json",
]:
    print(" -", os.path.join(DUMPS_DIR, filename))

Saved:
 - ../Dumps/X_train.parquet
 - ../Dumps/X_test.parquet
 - ../Dumps/y_train.parquet
 - ../Dumps/y_test.parquet
 - ../Dumps/preprocessor_template.pkl
 - ../Dumps/skf.pklfeature_metadata.json
